In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("..")

In [38]:
import numpy as np
import torch

In [4]:
import bayesgpt

In [5]:
from bayesgpt.networks import BayesGPT, BayesGPTv1

In [6]:
from bayesgpt.simulators.model_family import NestedModelFamily
from bayesgpt.simulators.benchmarks.ddms import DDM
from bayesgpt.simulators.benchmarks.ddms.ddm_priors import ddm_baseline_priors
from bayesgpt.adapters import Adapter

In [7]:
ddm_family = NestedModelFamily(
    name="ddm",
    model=DDM(),
    prior_fun=ddm_baseline_priors(),
    regressed_params=["v", "a", "tau"],
    mask_randomizer_kwargs=dict(
        free_intrinsics=["v", "a", "tau"],
        fixed_intrinsics=["s_v", "s_tau"],
        fixed_values={"s_v": 0, "s_tau": 0},
    )
)

In [8]:
sample_kwargs = {
    'min_num_regressors': 2,
    "max_num_regressors": 2,
    "max_num_categories": 2,
    "fixed_config": True
}

samples = ddm_family.batch_sample(
    batch_size=20,
    num_obs=50,
    flatten_param_outputs=True,
    **sample_kwargs
)

adapter = Adapter()

/home/radevs/anaconda3/envs/bf/lib/python3.11/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [9]:
for k, v in samples.items():
    print(k, v.shape if isinstance(v, np.ndarray) else v)

model_names ['ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm']
design_configs [{'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['v', 'a', 'tau']}, {'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['v', 'a', 'tau']}, {'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['v', 'a', 'tau']}, {'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['v', 'a', 'tau']}, {'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['v', 'a', 'tau']}, {'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['v', 'a', 'tau']}, {'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['v', 'a', 'tau']}, {'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['v', 'a', 'tau']}, {'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['v',

In [10]:
# Adapt
adapted = adapter.adapt(samples, intrinsic_params=ddm_family.intrinsic_params)

In [11]:
adapted["param_indices"].shape

torch.Size([20, 15, 1])

In [12]:
p = adapted["param_matrices"][..., None]

In [13]:
p.device

device(type='cuda', index=0)

In [34]:
bayesgpt = BayesGPT(encoder_input_dim=5, encoder_num_layers=4, decoder_num_layers=4, seed_dim=64, num_seeds=10)
# bayesgpt = BayesGPTv1(encoder_input_dim=5, encoder_num_layers=4, decoder_num_layers=4, seed_dim=64, num_seeds=10)
bayesgpt = bayesgpt.to("cuda")

In [35]:
print(adapted['input_data'].shape)
print(p.shape)

torch.Size([20, 50, 5])
torch.Size([20, 15, 1])


In [36]:
adapted['param_indices'].shape

torch.Size([20, 15, 1])

In [39]:
pred_velocity, target_velocity = bayesgpt(
    p,
    adapted['input_data'],
    adapted['param_indices'],
    adapted['regressor_indices'],
    adapted['param_masks']
)

bayesgpt.compute_loss(pred_velocity, target_velocity, adapted['param_masks'])

tensor(2.3402, device='cuda:0', grad_fn=<MeanBackward0>)

In [41]:
with torch.no_grad():
    samples = bayesgpt.sample(
        adapted['input_data'],
        adapted['param_indices'],
        adapted['regressor_indices'],
        adapted['param_masks']
    )

In [44]:
samples

tensor([[[-3.6955e-01],
         [ 6.8619e-01],
         [ 4.4415e-01],
         [-9.6626e-01],
         [-8.4958e-01],
         [-9.5383e-02],
         [-1.3134e+00],
         [-1.1936e+00],
         [ 1.7177e-01],
         [-1.1678e+00],
         [ 4.2169e-01],
         [ 1.9421e-01],
         [-3.1242e-01],
         [-8.4607e-01],
         [ 3.6199e-01]],

        [[ 2.2797e-01],
         [ 8.4182e-01],
         [-1.1547e+00],
         [-1.8791e-01],
         [-2.0094e+00],
         [ 2.7696e-01],
         [-9.7477e-01],
         [-1.5265e+00],
         [ 1.2484e+00],
         [-3.4656e-01],
         [ 6.4225e-01],
         [ 1.5007e-01],
         [-4.6059e-01],
         [-1.8097e+00],
         [-7.5748e-01]],

        [[-5.3284e-01],
         [-1.5512e+00],
         [-1.1952e-01],
         [-9.3970e-02],
         [ 4.1004e-01],
         [ 3.0430e-01],
         [ 6.6514e-01],
         [-2.0468e+00],
         [ 2.9870e-01],
         [ 2.2722e+00],
         [-1.7546e+00],
         [-1